# LLM Enrichers: AI-Powered Data Enhancement

Enrichers use LLMs (via Amazon Bedrock) to add intelligence to the pipeline — automatically
classifying, extracting, summarizing, or translating field values before graph construction.

| Enricher | Purpose |
|----------|--------|
| `BedrockEnricherPlugin` | Call any Bedrock model to enrich a field (classification, extraction, summarization) |
| `LLMEnricherPlugin` | Generic LLM enrichment (provider-agnostic interface) |
| `LanguageEnricherProvider` | Detect and tag the language of text fields |
| `RemediationFactoryProvider` | Automated data quality remediation via LLM |

## Why Enrichers Matter

Structured data often has fields that *could* contain more intelligence:
- A `description` field → extract entities, classify category, detect sentiment
- A `title` field → generate keywords, infer topic hierarchy
- A multi-language dataset → detect language per record for downstream routing
- Data quality issues → LLM can fix formatting, spelling, classification errors

Enrichers add these computed properties to records *before* graph construction,
so the resulting nodes carry AI-derived metadata alongside the source data.

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph`
- `pip install boto3` (for Bedrock access)
- AWS credentials with Bedrock model access (Claude, Titan, etc.)
- No Neptune required — enrichment runs locally, outputs to records

## Configuration

In [ ]:
import os

# Bedrock model to use for enrichment
MODEL_ID = os.environ.get('BEDROCK_MODEL_ID', 'anthropic.claude-3-haiku-20240307-v1:0')
AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')

print(f'Model: {MODEL_ID}')
print(f'Region: {AWS_REGION}')

## 1. Language Detection Enricher

Automatically detect the language of text fields. Useful for:
- Multi-language datasets that need routing to different processing pipelines
- Adding a `language` property to nodes for filtered queries
- Identifying data quality issues (unexpected languages in a monolingual dataset)

In [ ]:
from graphrag_toolkit.document_graph.transform.enrichers.language_enricher_provider import LanguageEnricherProvider
from graphrag_toolkit.document_graph.transform.transformer_provider_config import TransformerProviderConfig

config = TransformerProviderConfig(name='lang_detect', args={'field': 'description'})
enricher = LanguageEnricherProvider(config)

test_records = [
    {'id': '1', 'description': 'This is a security incident report from the SOC team.'},
    {'id': '2', 'description': 'Dies ist ein Sicherheitsvorfall-Bericht vom SOC-Team.'},
    {'id': '3', 'description': 'Ceci est un rapport incident de securite.'},
    {'id': '4', 'description': 'Este es un informe de incidente de seguridad.'},
]

result = enricher.transform(test_records)

print('Language detection results:')
for r in result:
    lang = r.get('language', r.get('detected_language', '?'))
    print(f'  [{lang}] {r["description"][:50]}...')

## 2. Bedrock Enricher (LLM-Powered Classification)

Use Amazon Bedrock to classify, extract, or summarize field values.
The enricher calls the model with a configurable prompt template and
adds the LLM's response as a new field on each record.

**Example**: Classify security alerts by severity based on their description.

In [ ]:
from graphrag_toolkit.document_graph.transform.enrichers.bedrock_enricher_plugin import BedrockEnricherPlugin

# Configure the enricher with a classification prompt
config = TransformerProviderConfig(name='classify_severity', args={
    'field': 'description',
    'output_field': 'ai_severity',
    'model_id': MODEL_ID,
    'region': AWS_REGION,
    'prompt_template': (
        'Classify the severity of this security alert as one of: '
        'CRITICAL, HIGH, MEDIUM, LOW, INFORMATIONAL.\n\n'
        'Alert: {value}\n\n'
        'Respond with only the severity level, nothing else.'
    )
})

enricher = BedrockEnricherPlugin(config)

test_records = [
    {'id': 'ALT-001', 'description': 'Unauthorized root login attempt from unknown IP 203.0.113.42 with brute force pattern detected'},
    {'id': 'ALT-002', 'description': 'SSL certificate expiring in 30 days for internal monitoring dashboard'},
    {'id': 'ALT-003', 'description': 'Data exfiltration detected: 50GB transferred to external S3 bucket in last hour'},
    {'id': 'ALT-004', 'description': 'User logged in from new device, MFA verified successfully'},
]

# Note: This requires active Bedrock access. Will fail without credentials.
try:
    result = enricher.transform(test_records)
    print('Bedrock classification results:')
    for r in result:
        print(f'  [{r.get("ai_severity", "?")}] {r["id"]}: {r["description"][:60]}...')
except Exception as e:
    print(f'Bedrock not available (expected if no AWS credentials): {e}')
    print('\nIn production, this would add an ai_severity field to each record:')
    print('  [CRITICAL] ALT-001: Unauthorized root login...')
    print('  [LOW]      ALT-002: SSL certificate expiring...')
    print('  [CRITICAL] ALT-003: Data exfiltration detected...')
    print('  [INFORMATIONAL] ALT-004: User logged in from new device...')

## 3. Bedrock Enricher (Entity Extraction)

Use an LLM to extract structured entities from free-text fields.
This bridges unstructured descriptions into structured graph properties.

In [ ]:
config = TransformerProviderConfig(name='extract_entities', args={
    'field': 'description',
    'output_field': 'extracted_entities',
    'model_id': MODEL_ID,
    'region': AWS_REGION,
    'prompt_template': (
        'Extract all IP addresses, usernames, and AWS resource ARNs from this text.\n'
        'Return as comma-separated values. If none found, return "none".\n\n'
        'Text: {value}\n\n'
        'Entities:'
    )
})

enricher = BedrockEnricherPlugin(config)

test_records = [
    {'id': 'EVT-001', 'description': 'User admin_jane accessed arn:aws:s3:::prod-data from 10.0.1.55'},
    {'id': 'EVT-002', 'description': 'Lambda function arn:aws:lambda:us-east-1:123456789012:function:processor timed out'},
]

try:
    result = enricher.transform(test_records)
    print('Entity extraction results:')
    for r in result:
        print(f'  {r["id"]}: {r.get("extracted_entities", "?")}')
except Exception as e:
    print(f'Bedrock not available: {e}')
    print('\nIn production, this would extract:')
    print('  EVT-001: admin_jane, arn:aws:s3:::prod-data, 10.0.1.55')
    print('  EVT-002: arn:aws:lambda:us-east-1:123456789012:function:processor')

## 4. Enricher in a Full Pipeline

Enrichers slot into the transform stage of the pipeline. They run after normalizers
(clean data) and before graph transformers (build structure):

```
Extract (CSV/Parquet/JSON)
    ↓
Ingest (column select, row filter)
    ↓
Normalize (whitespace, nulls, case)
    ↓
★ ENRICH (LLM classification, entity extraction, language detection)
    ↓
Graph Transform (row→node, infer edges)
    ↓
Construct (Cypher generation)
    ↓
Load (Neptune write)
```

The enriched fields become node properties — queryable in Neptune alongside
the original source data.

In [ ]:
from graphrag_toolkit.document_graph.transform.normalizers.normalize_whitespace_provider import NormalizeWhitespaceProvider
from graphrag_toolkit.document_graph.transform.graph_transformers.row_to_node import RowToNodeTransformer
from graphrag_toolkit.document_graph import Node
from graphrag_toolkit.document_graph.graph_build.cypher_builder import node_to_cypher

# Simulated pipeline with enrichment
raw_records = [
    {'id': 'ALT-001', 'description': '  Unauthorized root login from 203.0.113.42  ', 'source': 'guardduty'},
    {'id': 'ALT-002', 'description': '  SSL cert expiring in 30 days  ', 'source': 'config'},
]

# Step 1: Normalize
ws_config = TransformerProviderConfig(name='ws', args={})
normalized = NormalizeWhitespaceProvider(ws_config).transform(raw_records)
print('After normalize:', [r['description'] for r in normalized])

# Step 2: Enrich (simulated — add classification without Bedrock)
for r in normalized:
    # In production: enricher.transform(normalized) calls Bedrock
    r['ai_severity'] = 'CRITICAL' if 'unauthorized' in r['description'].lower() else 'LOW'
    r['ai_category'] = 'access_violation' if 'login' in r['description'].lower() else 'maintenance'
print('After enrich:', [{k: v for k, v in r.items() if k.startswith('ai_')} for r in normalized])

# Step 3: Graph transform + Cypher
for r in normalized:
    node = Node(id=r['id'], labels=['SecurityAlert'], properties=r)
    cypher, params = node_to_cypher(node, tenant_id='enriched')
    print(f'\nNode {r["id"]}:')
    print(f'  Cypher: {cypher}')
    print(f'  Props include ai_severity={r["ai_severity"]}, ai_category={r["ai_category"]}')

print('\n✅ Pipeline with enrichment: normalize → enrich → graph build')

## Summary

| Enricher | Input | Output | Requires |
|----------|-------|--------|----------|
| Language Detection | text field | `language` field | No cloud (local detection) |
| Bedrock Classification | text field | classification label field | Bedrock access |
| Bedrock Entity Extraction | text field | extracted entities field | Bedrock access |
| Remediation Factory | dirty field | cleaned field | Bedrock access |

Enrichers transform dumb data into smart nodes — enabling queries like:
- "Find all CRITICAL security alerts" (via `ai_severity` property)
- "Show alerts involving IP 203.0.113.42" (via `extracted_entities` property)
- "Filter to English-language reports only" (via `language` property)